# ЭДКТ: обучение агента размещения контейнеров (PPO с маской действий)

**Что здесь происходит.** Агент учится выбирать ячейку для контейнера на площадке ЭДКТ (3 яруса × 3 колонны × 30 рядов, пути у рядов 5/15/25).
Он выбирает одну из 8 лучших по формуле автора ячеек и учится на последствиях: перестановки при выдаче, отказы в приёме, стоимость операций.

**Данные синтетические.** Поток контейнеров генерирует имитационная модель (пуассоновский поток, 60 % — 20 фут., сроки хранения 6–480 ч, ошибка прогноза даты 0–24 ч).
Это демонстрация функциональности: при сотрудничестве с терминалом агент дообучается на его реальных данных.

**Откуда правила.** Геометрия площадки, штабелирование (40 фут. — пара рядов, только на такой же размер) и тариф берутся из `app/placement.py` — того же модуля, что работает в программе.
Маска действий не даёт агенту выбрать недопустимую ячейку.

**Воспроизводимость.** seed = 7, все параметры и SHA-256 весов пишутся в `out/train_info.json`.

Файлы: `rl/yard_sim.py` — симулятор, `rl/env.py` — среда Gymnasium, `app/agent/features.py` — признаки, `app/agent/reward.py` — награда (пишет автор), `rl/train.py` — обучение, `rl/export.py` — экспорт весов в numpy, `rl/evaluate.py` — проверка на новых seed.

In [ ]:
!pip install -q sb3-contrib gymnasium 2>&1 | tail -1
import glob, os, shutil
src = os.path.dirname(glob.glob('/kaggle/input/**/rl/train.py', recursive=True)[0])
src = os.path.dirname(src)
shutil.copytree(src, '/kaggle/working/edkt', dirs_exist_ok=True)
os.chdir('/kaggle/working/edkt')
print('код из', src); print(open('app/agent/reward.py').read())

## Формула v2
Веса формулы автора подобраны автоматически (Optuna, 80 попыток, обучающие seed 0–3). Результат — `rl/formula_v2.json`; по ней агент отбирает 8 кандидатов.

In [ ]:
import json; print(json.dumps(json.load(open('rl/formula_v2.json')), ensure_ascii=False, indent=1))

## Обучение
MaskablePPO, сеть 128×128 (Tanh), γ = 0.995 (последствия решения — перестановка — проявляются через дни), 4 параллельные среды, 3 млн шагов.

In [ ]:
!python rl/train.py --steps 3000000 --envs 4 --seed 7 --out /kaggle/working/out 2>&1 | grep -E 'total_timesteps|ep_rew_mean|minutes|sha256' | tail -40

## Проверка на новых seed (100–114)
Сравнение: случайная ячейка, ближайшая к пути, формула автора (v1), формула v2, агент. Загрузка ≈ 60 / 72 / 80 % (поток 0,6 / 0,75 / 0,9 конт./ч), ошибка прогноза ±12 ч.

In [ ]:
!python rl/evaluate.py --policy /kaggle/working/out/agent_policy.npz --out /kaggle/working/out/eval.json

In [ ]:
print(open('/kaggle/working/out/train_info.json').read())